# Job Market Intelligence Dashboard
## Skill Trends Analysis

**Dataset:** `lukebarousse/data_jobs` (Hugging Face) — US Data Analyst roles (2023)  
**Goal:** Understand *which* skills are rising or falling in demand, which skills appear together, and what the "optimal" skill set looks like for a Data Analyst.

---
### Sections
1. Setup & data loading  
2. Monthly skill demand trends  
3. Skill co-occurrence analysis  
4. Optimal skills — salary vs. demand scatter  
5. Technology category breakdown  
6. Skill growth momentum report

In [ ]:
# ── 1. Setup & Data Loading ──────────────────────────────────────────────────
import ast
import os
import sys
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore")

# ── Project root so utils/helpers.py is importable ───────────────────────────
_NB_DIR = os.path.abspath(".")
_ROOT   = os.path.dirname(_NB_DIR) if os.path.basename(_NB_DIR) == "analysis" else _NB_DIR
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

sns.set_theme(style="ticks", palette="tab10")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13})

# ── Load dataset ──────────────────────────────────────────────────────────────
dataset = load_dataset("lukebarousse/data_jobs")
df_all = dataset["train"].to_pandas()
df_all["job_posted_date"] = pd.to_datetime(df_all["job_posted_date"])
df_all["job_skills"] = df_all["job_skills"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# Filter: US Data Analysts (exact title for richest skill coverage)
df_da = df_all[
    (df_all["job_title_short"] == "Data Analyst") &
    (df_all["job_country"] == "United States")
].copy()

df_da["month_no"]  = df_da["job_posted_date"].dt.month
df_da["month_str"] = df_da["job_posted_date"].dt.strftime("%b")

print(f"US Data Analyst postings: {len(df_da):,}")
print(f"With skills listed:       {df_da['job_skills'].apply(len).gt(0).sum():,}")
print(f"With salary disclosed:    {df_da['salary_year_avg'].notna().sum():,}")

---
## 2. Monthly Skill Demand Trends

How does the demand for the top 5 Data Analyst skills evolve month-by-month throughout 2023? We express demand as the **percentage of job postings** that mention each skill — this normalises for variation in monthly posting volume.

In [ ]:
# ── 2. Monthly skill demand as % of postings ─────────────────────────────────
df_da_exp = df_da.explode("job_skills").dropna(subset=["job_skills"]).copy()

# Pivot table: rows=month, cols=skill, values=skill count
pivot = df_da_exp.pivot_table(
    index="month_no",
    columns="job_skills",
    aggfunc="size",
    fill_value=0,
)

# Sort columns by total demand, keep top 10
pivot.loc["_total_", :] = pivot.sum()
pivot = pivot[pivot.loc["_total_"].sort_values(ascending=False).index]
pivot = pivot.drop(index="_total_")

# Monthly totals (pre-explode)
monthly_totals = df_da.groupby("month_no").size()

# Convert to % of monthly postings
pivot_pct = pivot.div(monthly_totals, axis=0) * 100
pivot_pct = pivot_pct.reset_index()
pivot_pct["month_label"] = pivot_pct["month_no"].apply(
    lambda m: pd.to_datetime(str(m), format="%m").strftime("%b")
)
pivot_pct = pivot_pct.set_index("month_label").drop(columns="month_no")

TOP_N = 5
df_plot = pivot_pct.iloc[:, :TOP_N]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df_plot, dashes=False, palette="tab10", linewidth=2.2, ax=ax)
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title(f"Top {TOP_N} Skill Demand Trends — US Data Analysts (2023)", fontweight="bold")
ax.set_xlabel("Month (2023)")
ax.set_ylabel("% of Job Postings Mentioning Skill")
ax.get_legend().remove()

# Annotate skill names at right edge
for i, skill in enumerate(df_plot.columns):
    last_val = df_plot.iloc[-1, i]
    ax.text(len(df_plot) - 0.7, last_val, f" {skill}", va="center", fontsize=9,
            color=sns.color_palette("tab10")[i])

sns.despine()
plt.tight_layout()
plt.show()

---
## 3. Skill Co-occurrence Analysis

Which skills are most frequently **required together** in the same job posting? Understanding skill combinations helps candidates plan upskilling roadmaps strategically.

We build a symmetric co-occurrence matrix for the top 15 skills, then visualise it as a heatmap.

In [ ]:
# ── 3. Skill co-occurrence matrix ─────────────────────────────────────────────
# Identify top N skills
TOP_CO = 15
top_skills_list = pivot.iloc[:, :TOP_CO].columns.tolist()

# Build co-occurrence matrix
cooc = pd.DataFrame(0, index=top_skills_list, columns=top_skills_list)

for skill_set in df_da["job_skills"]:
    relevant = [s for s in skill_set if s in top_skills_list]
    for s1, s2 in combinations(relevant, 2):
        cooc.loc[s1, s2] += 1
        cooc.loc[s2, s1] += 1

# Fill diagonal with individual skill frequency
for skill in top_skills_list:
    cooc.loc[skill, skill] = (df_da_exp["job_skills"] == skill).sum()

# ── Plot ──────────────────────────────────────────────────────────────────────
mask = np.zeros_like(cooc, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True   # upper-right triangle hidden

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    cooc, mask=mask, annot=True, fmt="d",
    cmap="Blues", linewidths=0.4, linecolor="white",
    ax=ax
)
ax.set_title(
    f"Skill Co-occurrence — Top {TOP_CO} Skills in US Data Analyst Postings (2023)",
    fontweight="bold", pad=12
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Top 10 skill pairs
pair_rows = []
for s1, s2 in combinations(top_skills_list, 2):
    pair_rows.append({"skill_pair": f"{s1} + {s2}", "co_occurrences": cooc.loc[s1, s2]})

top_pairs = pd.DataFrame(pair_rows).nlargest(10, "co_occurrences").reset_index(drop=True)
print("Top 10 skill combinations:")
print(top_pairs.to_string(index=False))

---
## 4. Optimal Skills — Salary vs. Demand

The **"optimal" skill** is one that is:
- **Frequently required** → high demand / job-market relevance  
- **Well-compensated** → high median salary

We plot each skill as a point in (demand %, median salary) space. Skills in the **top-right quadrant** are the most strategically valuable to learn.

> Threshold: only skills mentioned in > 5 % of postings are shown.

In [ ]:
# ── 4. Optimal skills scatter ────────────────────────────────────────────────
df_da_sal = df_da.dropna(subset=["salary_year_avg"]).explode("job_skills").dropna(subset=["job_skills"])
total_da = len(df_da.dropna(subset=["salary_year_avg"]))

skill_agg = (
    df_da_sal
    .groupby("job_skills")["salary_year_avg"]
    .agg(skill_count="count", median_salary="median")
    .reset_index()
)
skill_agg["skill_pct"] = skill_agg["skill_count"] / total_da * 100
skill_agg = skill_agg[skill_agg["skill_pct"] > 5].copy()

# Technology category mapping (simplified)
TECH_CAT = {
    "sql": "Databases", "excel": "Analyst Tools", "python": "Programming",
    "tableau": "Analyst Tools", "r": "Programming", "power bi": "Analyst Tools",
    "sas": "Analyst Tools", "powerpoint": "Analyst Tools", "word": "Analyst Tools",
    "oracle": "Databases", "sql server": "Databases",
    "spark": "Big Data", "hadoop": "Big Data",
    "azure": "Cloud", "aws": "Cloud", "go": "Programming",
    "javascript": "Programming", "looker": "Analyst Tools",
}
skill_agg["category"] = skill_agg["job_skills"].str.lower().map(TECH_CAT).fillna("Other")

# ── Plot ──────────────────────────────────────────────────────────────────────
try:
    from adjustText import adjust_text
    HAS_ADJUST = True
except ImportError:
    HAS_ADJUST = False

fig, ax = plt.subplots(figsize=(11, 7))
category_colors = {
    "Programming": "#1f77b4",
    "Databases": "#ff7f0e",
    "Analyst Tools": "#2ca02c",
    "Big Data": "#d62728",
    "Cloud": "#9467bd",
    "Other": "#8c564b",
}

for cat, grp in skill_agg.groupby("category"):
    ax.scatter(
        grp["skill_pct"], grp["median_salary"],
        label=cat, color=category_colors.get(cat, "gray"),
        s=90, zorder=3, edgecolors="white", linewidths=0.6
    )

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${int(v/1000)}K"))
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title("Most Optimal Skills for US Data Analysts — Salary vs. Demand (2023)",
             fontweight="bold")
ax.set_xlabel("% of Data Analyst Job Postings Requiring Skill")
ax.set_ylabel("Median Annual Salary (USD)")
ax.legend(title="Technology Category", loc="lower right")
ax.axhline(skill_agg["median_salary"].median(), color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(skill_agg["skill_pct"].median(),    color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

texts = [
    ax.text(r.skill_pct, r.median_salary, f" {r.job_skills}", fontsize=8)
    for _, r in skill_agg.iterrows()
]
if HAS_ADJUST:
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="->", color="gray", lw=0.6))

sns.despine()
plt.tight_layout()
plt.show()

---
## 5. Skill Growth Momentum

We compute the **month-over-month percentage change** in demand for the top 10 skills:

$$\Delta\% = \frac{v_t - v_{t-1}}{v_{t-1}} \times 100$$

This identifies the **fastest-growing** and fastest-declining skills over the year.

In [ ]:
# ── 5a. Month-over-month change for top 10 skills ─────────────────────────────
df_plot_top10 = pivot_pct.iloc[:, :10]

mom_change = df_plot_top10.pct_change() * 100   # month-over-month % change

fig, ax = plt.subplots(figsize=(12, 5))
for i, skill in enumerate(df_plot_top10.columns):
    ax.plot(
        df_plot_top10.index, mom_change[skill],
        marker="o", linewidth=1.8,
        label=skill, color=sns.color_palette("tab10")[i]
    )

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title("Month-over-Month Demand Change — Top 10 Skills (US Data Analysts, 2023)",
             fontweight="bold")
ax.set_xlabel("Month (2023)")
ax.set_ylabel("Δ% vs. Previous Month")
ax.legend(loc="upper right", fontsize=8, ncol=2)
plt.xticks(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Net growth summary (Dec vs. Jan) ──────────────────────────────────────
net_growth = (
    (df_plot_top10.iloc[-1] - df_plot_top10.iloc[0])
    / df_plot_top10.iloc[0].replace(0, np.nan)
    * 100
).sort_values()

colors = ["#d62728" if v < 0 else "#2ca02c" for v in net_growth]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(net_growth.index, net_growth.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title("Net Demand Change Jan → Dec 2023 — Top 10 Skills (US Data Analysts)",
             fontweight="bold")
ax.set_xlabel("Net % Change in Monthly Demand")
ax.set_ylabel("")
sns.despine()
plt.tight_layout()
plt.show()

rising  = net_growth[net_growth > 0].sort_values(ascending=False)
falling = net_growth[net_growth < 0].sort_values()
print("Fastest-growing skills:", rising.index[:3].tolist())
print("Fastest-declining skills:", falling.index[:3].tolist())

---
## 6. Skill Demand Summary Table

A ranked summary combining demand frequency, median salary, and net YoY growth — suitable for export or embedding in a portfolio report.

In [ ]:
# ── 6. Comprehensive skill summary ────────────────────────────────────────────
# All skills with count (entire year)
all_skill_freq = df_da_exp["job_skills"].value_counts().reset_index()
all_skill_freq.columns = ["skill", "job_count"]
all_skill_freq["pct_of_postings"] = (all_skill_freq["job_count"] / len(df_da) * 100).round(1)

# Merge salary stats
sal_stats = (
    df_da_sal
    .groupby("job_skills")["salary_year_avg"]
    .agg(median_salary="median", salary_count="count")
    .reset_index()
    .rename(columns={"job_skills": "skill"})
)

summary = all_skill_freq.merge(sal_stats, on="skill", how="left")
summary["median_salary"] = summary["median_salary"].round(0)
summary["salary_count"]  = summary["salary_count"].fillna(0).astype(int)
summary = summary.sort_values("job_count", ascending=False).reset_index(drop=True)
summary.index += 1
summary.index.name = "rank"

# Save to processed folder
processed_dir = os.path.join(_ROOT, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)
summary.to_csv(os.path.join(processed_dir, "skill_summary.csv"))
print(f"Skill summary saved → {processed_dir}/skill_summary.csv")
print()
summary.head(20).style \
    .format({"pct_of_postings": "{:.1f}%", "median_salary": "${:,.0f}"}) \
    .background_gradient(subset=["job_count"], cmap="Blues") \
    .background_gradient(subset=["median_salary"], cmap="Greens")